In [3]:
!pip install fastmcp


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install pydantic_ai

  Obtaining dependency information for pydantic_ai from https://files.pythonhosted.org/packages/b8/c6/b85d5d8da15c616374a5bf566aaa5ba13142b0024fddfe97ac288641792b/pydantic_ai-1.104.0-py3-none-any.whl.metadata
  Obtaining dependency information for pydantic-ai-slim[ag-ui,anthropic,bedrock,cli,cohere,evals,fastmcp,google,groq,huggingface,logfire,mcp,mistral,openai,retries,spec,temporal,ui,vertexai,xai]==1.104.0 from https://files.pythonhosted.org/packages/6a/1d/c03cecf9c48040f750c6e5b4f027fb4935cc41d9b76f1217af7731f4dc2b/pydantic_ai_slim-1.104.0-py3-none-any.whl.metadata
  Obtaining dependency information for genai-prices>=0.0.56 from https://files.pythonhosted.org/packages/81/35/ce64112dcc6f406b3e290dcf57a97acfa2b7d3d0391979219cb9d4a9db6d/genai_prices-0.0.62-py3-none-any.whl.metadata
  Using cached genai_prices-0.0.62-py3-none-any.whl.metadata (7.1 kB)
  Obtaining dependency information for griffelib>=2.0 from https://files.pythonhosted.org/packages/11/8c/c9138d881c79aa0ea9ed83cbd58d5ca

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepeval 2.8.9 requires anthropic<0.50.0,>=0.49.0, but you have anthropic 0.105.2 which is incompatible.
deepeval 2.8.9 requires google-genai<2.0.0,>=1.9.0, but you have google-genai 2.7.0 which is incompatible.
fastembed 0.4.0 requires huggingface-hub<1.0,>=0.20, but you have huggingface-hub 1.17.0 which is incompatible.
google-auth-oauthlib 1.2.3 requires google-auth<2.42.0,>=2.15.0, but you have google-auth 2.53.0 which is incompatible.
langchain-core 0.3.51 requires packaging<25,>=23.2, but you have packaging 25.0 which is incompatible.
langchain-google-genai 2.1.2 requires google-ai-generativelanguage<0.7.0,>=0.6.16, but you have google-ai-generativelanguage 0.6.15 which is incompatible.
langchain-openai 0.3.12 requires openai<2.0.0,>=1.68.2, but you have openai 2.38.0 which is incompatible.
mlflow 3.12.0 req

In [ ]:
import os
import json
import sys

# 1. APPLY WINDOWS JUPYTER SUBPROCESS PATCH
# This restores the OS-level stderr file descriptor, preventing the 'fileno' error.
sys.stderr = sys.__stderr__
import random
import asyncio
import textwrap
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Any, Dict, List, Callable, Coroutine
import pandas as pd
import mlflow
import mlflow.entities
import mlflow.data
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
import random
import functools
from pydantic_ai import Agent, ModelRetry
from pydantic_ai.agent import RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.profiles import ModelProfile
from pydantic_ai.settings import ModelSettings
from functools import partial
import matplotlib.pyplot as plt
import seaborn as sns

from mlflow.genai.scorers import (
    Correctness,
    Guidelines,
    ExpectationsGuidelines,
    scorer,
    ScorerSamplingConfig,
)
from mlflow.metrics.genai import (
    EvaluationExample,
    faithfulness,
    answer_correctness,
)
from mlflow.data.pandas_dataset import PandasDataset
from mlflow.genai.datasets import create_dataset, get_dataset
from mlflow.genai import make_judge

from mlflow.entities import Feedback
from mlflow import MlflowClient

from pydantic_ai import Agent,capture_run_messages
from pydantic_ai.mcp import MCPToolset
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.profiles import ModelProfile
from pydantic_ai.settings import ModelSettings

In [ ]:
import litellm
from dotenv import load_dotenv
load_dotenv()

litellm.set_verbose = True  

In [ ]:
#!MLFLOW_SERVER_ALLOWED_HOSTS="*" MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*" mlflow server --port=8050 --host=0.0.0.0 --backend-store-uri sqlite:///mlflow.db
!mlflow server --backend-store-uri sqlite:///mlflow.db --host=127.0.0.1 --port=8050


Der Befehl "mlflow" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


In [ ]:
def multiply_by_two(x, y): #pdf
    """ArithmeticError: This function is designed to fail for demonstration purposes."""

    return x * y # viel text

In [ ]:
text = multiply_by_two(3, 4)

In [ ]:

model_id = "gemma3:27b"#"gemma3:27b" #"Qwen/Qwen3-VL-30B-A3B-Instruct-FP8" #"RedHatAI/Qwen3-32B-quantized.w4a16"
judge_uri = f"openai:/{model_id}"



In [5]:

mlflow.set_tracking_uri("http://localhost:8050") 
exp_name = "Job_CV_Evaluation"

dataset_job_description = "eval_datset_job_description"
dataset_CV_description = "eval_datset_CV_description"
dataset_Matcher_description = "eval_datset_Matcher_description"

mlflow.set_experiment(exp_name)

experiment = mlflow.get_experiment_by_name(exp_name)
exp_id = experiment.experiment_id
#mlflow.set_tracking_uri("sqlite:///mlflow.db") 




In [36]:

def load_jobbank_jsonl(path):
    """
    Loads a JSONL file with fields:
    - url
    - title
    - full_page_text
    into a pandas DataFrame.
    """
    df = pd.read_json(path, lines=True)
    return df




In [37]:
df = load_jobbank_jsonl("data_collection/jobbank_jobs.jsonl")
df

,url,title,full_page_text
0,https://www.jobbank.gc.ca/jobsearch/jobposting...,software engineer,"software engineer - Mississauga, ON - Job post..."
1,https://www.jobbank.gc.ca/jobsearch/jobposting...,software engineer,"software engineer - Toronto, ON - Job posting ..."
2,https://www.jobbank.gc.ca/jobsearch/jobposting...,software engineer,"software engineer - Hamilton, ON - Job posting..."
3,https://www.jobbank.gc.ca/jobsearch/jobposting...,lead software engineer,"lead software engineer - Richmond Hill, ON - J..."
4,https://www.jobbank.gc.ca/jobsearch/jobposting...,software engineer,"software engineer - Vancouver, BC - Job postin..."
...,...,...,...
226,https://www.jobbank.gc.ca/jobsearch/jobposting...,graphic designer,"graphic designer - Calgary, AB - Job posting -..."
227,https://www.jobbank.gc.ca/jobsearch/jobposting...,graphic designer,"graphic designer - Coquitlam, BC - Job posting..."
228,https://www.jobbank.gc.ca/jobsearch/jobposting...,graphic designer,"graphic designer - Québec, QC - Job posting - ..."
229,https://www.jobbank.gc.ca/jobsearch/jobposting...,graphic designer,"graphic designer - Burnaby, BC - Job posting -..."


In [38]:
[# according to our company who is a .....
    "Software Engineer", "Software Developer", "Full Stack Developer",
    "Backend Developer", "Frontend Developer", "Data Scientist",
    "Machine Learning Engineer", "AI Engineer", "DevOps Engineer",
    "Cloud Engineer", "Data Analyst", "Business Intelligence Analyst",
    "QA Engineer", "Test Automation Engineer", "iOS Developer",
    "Android Developer", "UI Designer", "UX Designer", "Product Designer",
    "Cybersecurity Analyst", "Python Developer", "Data Engineer",
    "Network Engineer", "Cloud Architect", "Systems Engineer",
    "Java Developer", ".NET Developer", "Web Developer", "SDET",
    "Solutions Architect", "Big Data Specialist", "Fintech Engineer",
    "AI Prompt Engineer", "Blockchain Developer", "Robotics Engineer",
    "Javascript Developer", "AR Developer", "VR Developer",
    "IoT Engineer", "Ethical Hacker", "SRE", "Game Developer",

    "Product Manager", "Project Manager", "Marketing Specialist",
    "Digital Marketing Specialist", "SEO Specialist", "Content Writer",
    "Copywriter", "Business Analyst", "Operations Manager",
    "Sales Executive", "Technical Writer", "Market Research Analyst",
    "Graphic Designer"
]




['Software Engineer',
 'Software Developer',
 'Full Stack Developer',
 'Backend Developer',
 'Frontend Developer',
 'Data Scientist',
 'Machine Learning Engineer',
 'AI Engineer',
 'DevOps Engineer',
 'Cloud Engineer',
 'Data Analyst',
 'Business Intelligence Analyst',
 'QA Engineer',
 'Test Automation Engineer',
 'iOS Developer',
 'Android Developer',
 'UI Designer',
 'UX Designer',
 'Product Designer',
 'Cybersecurity Analyst',
 'Python Developer',
 'Data Engineer',
 'Network Engineer',
 'Cloud Architect',
 'Systems Engineer',
 'Java Developer',
 '.NET Developer',
 'Web Developer',
 'SDET',
 'Solutions Architect',
 'Big Data Specialist',
 'Fintech Engineer',
 'AI Prompt Engineer',
 'Blockchain Developer',
 'Robotics Engineer',
 'Javascript Developer',
 'AR Developer',
 'VR Developer',
 'IoT Engineer',
 'Ethical Hacker',
 'SRE',
 'Game Developer',
 'Product Manager',
 'Project Manager',
 'Marketing Specialist',
 'Digital Marketing Specialist',
 'SEO Specialist',
 'Content Writer',
 'C

In [ ]:


model = OpenAIChatModel(
    os.getenv("MODEL_ID_UNI_GREIFSWALD"),
    provider=OpenAIProvider(
        base_url= os.getenv("OPENAI_API_BASE_UNI_GREIFSWALD"),
        api_key=os.getenv("OPENAI_API_KEY_UNI_GREIFSWALD"), 
        #http_client=subagent_debug_client
    ),
    profile=ModelProfile(
        default_structured_output_mode='tool',
        supports_json_schema_output=False,
    ),
)
extra_body_dict = json.loads(os.getenv("LITELLM_EXTRA_BODY_UNI_GREIFSWALD"))
settings =  ModelSettings(
    extra_body=extra_body_dict
)
#extraction_agent = Agent(
#    model,
#    model_settings=settings,
#    #system_prompt=new_template,
#    #deps_type=ArticleReviewInput,
#    #output_type=ArticleReviewOutput,
#    retries=3 
#)


In [40]:
from schemas import JDExtractionOutput, ExperienceInfo
from typing import List, Dict, Any


In [41]:
import json

path = "job_main_domains.json"
path_2 = "jobs_details.json"

with open(path, "r", encoding="utf-8") as f:
    details_about_job_domains = json.load(f)

with open(path_2, "r", encoding="utf-8") as f:
    details_about_job = json.load(f)


In [42]:
JOBS_PER_ROLE_DOMAIN = details_about_job_domains["jobs_per_role_domain"]
ROLE_DOMAINS = details_about_job_domains["automotive_role_domains"]
JOBS_DETAILS_CONTEXT = details_about_job

In [43]:
ROLE_DOMAINS

[{'name': 'Vehicle_Tech_and_Software_Defined_Vehicles',
  'description': "This domain governs the physical vehicle's computing core, transitioning traditional mechanical cars into Software-Defined Vehicles (SDVs). It encompasses real-time embedded systems programming for Electronic Control Units (ECUs) and microcontroller architectures. Key sub-fields include Advanced Driver Assistance Systems (ADAS) and Autonomous Driving, which rely on AI and Machine Learning for complex tasks like computer vision, path planning, and sensor fusion (LiDAR, radar, cameras). It also includes Hardware-in-the-Loop (HIL) testing to safely validate code against simulated physical hardware, and strict adherence to functional safety standards (ISO 26262). Additionally, it integrates advanced cockpit graphics (such as AR/VR head-up displays and gaming engines for 3D dashboard rendering) and physical assembly automation through robotics engineering on the manufacturing floor."},
 {'name': 'Cloud_Data_and_Connec

In [44]:
def generate_reverse_lookup() -> Dict[str, Dict[str, str]]:
    """
    Creates a case-insensitive lookup table mapping target titles 
    to their domain, internal key name, and proper display format.
    """
    lookup = {}
    for domain, jobs in JOBS_PER_ROLE_DOMAIN.items():
        for item in ROLE_DOMAINS:
            domain_description = item["description"]
        for job_key, display_name in jobs.items():
            normalized_title = display_name.lower().strip()
            lookup[normalized_title] = {
                "domain_name": domain,
                "domain_description": domain_description,
                "job_key": job_key,
                "job_meaning": JOBS_DETAILS_CONTEXT.get(job_key, "No custom context available."),
                "display_name": display_name
            }
    return lookup

LOOKUP_MAP = generate_reverse_lookup()

In [45]:
LOOKUP_MAP

{'systems engineer': {'domain_name': 'Vehicle_Tech_and_Software_Defined_Vehicles',
  'domain_description': "This domain focuses on driving commercial growth, managing customer relationships, and modernizing automotive business models. Key sub-fields include market research and business intelligence, which leverage data analytics to forecast consumer demand and optimize vehicle pricing. It covers both digital and traditional marketing operations (SEO, copywriters, and graphic designers) to build brand awareness for new vehicle launches. Commercial operations handle complex dealer network management systems and fleet sales accounts. Furthermore, as vehicles become more software-driven, this domain manages 'Features-on-Demand' (FoD) and digital subscription services, handling the business strategy, customer billing, and digital storefronts required for post-purchase software upgrades (such as range extension or upgraded navigation packages).",
  'job_key': 'systems_engineer',
  'job_meani

# creating ground truth dataset for job description extraction

In [46]:
def extract_and_sample_diverse_jobs(df: pd.DataFrame, target_size: int = 20) -> pd.DataFrame:
    """
    Identifies job titles contained in the allowed taxonomy list.
    Extracts matches and ensures a balanced sample of exactly 20 across domains.
    """
    matched_rows = []
    
    for idx, row in df.iterrows():
        title_clean = str(row["title"]).lower().strip()
        if title_clean in LOOKUP_MAP:
            mapping_info = LOOKUP_MAP[title_clean]
            row_data = row.to_dict()
            # Enrich row with taxomony metadata
            row_data.update({
                "matched_display_title": mapping_info["display_name"],
                "target_domain": mapping_info["domain_name"],
                "domain_description": mapping_info["domain_description"],
                "job_key": mapping_info["job_key"],
                "job_meaning": mapping_info["job_meaning"]
            })
            matched_rows.append(row_data)
            
    candidates_df = pd.DataFrame(matched_rows)
    if candidates_df.empty:
        raise ValueError("No matching job titles from the list were found in the dataset.")
        
    # Implement a round-robin selector over unique domains to ensure diversity
    sampled_rows = []
    sampled_indices = set()
    unique_domains = list(candidates_df["target_domain"].unique())
    
    remaining_candidates = candidates_df.copy()
    
    while len(sampled_rows) < target_size and not remaining_candidates.empty:
        for domain in unique_domains:
            domain_pool = remaining_candidates[remaining_candidates["target_domain"] == domain]
            if not domain_pool.empty:
                selected_row = domain_pool.iloc[0]
                sampled_rows.append(selected_row.to_dict())
                sampled_indices.add(selected_row.name)
                # Drop selected row from pool to prevent duplication
                remaining_candidates = remaining_candidates.drop(selected_row.name)
            if len(sampled_rows) == target_size:
                break
                
    return pd.DataFrame(sampled_rows)

In [47]:
async def run_targeted_extraction(row: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Executes structural extraction using a localized schema definition,
    providing the specific context for the role.
    """
    # Dynamic context block constructed specifically for the matched role
    dynamic_prompt = f"""
    You are an expert recruitment system. Your task is to analyze the Job Description and extract structured attributes matching the schema requirements.

    To guide your extraction, our company defines this role and domain with the following specifications:

    Target Title: {row['matched_display_title']}
    Target Domain: {row['target_domain']}

    --- Domain Objectives ---
    {row['domain_description']}

    --- Specific Role Profile ---
    {row['job_meaning']}

    --- Extraction Task ---
    Extract attributes from the Job Description below. Interpret responsibilities, requirements, and systems in direct alignment with our Corporate Profile parameters for a {row['matched_display_title']}.

    Job Description Content:
    {row['full_page_text']}
    
    --- Strict Formatting & Fallback Rules ---
    You must populate EVERY field in the output schema. Omitting fields or using 'null' values is not allowed. 
    If the Job Description lacks information for a field, follow these fallback instructions:
    1. For all List/Array fields (`required_toolchains`, `compliance_and_standards`, `responsibilities`, `must_have`, `nice_to_have`): if no items are found, you MUST return an empty list: [].
    2. For the `years_of_experience` and `experience_level` fields: if they cannot be inferred, you MUST return an empty string: "".
    3. For `salary_range`: if not specified, you MUST return the string "Not Specified".
    4. Do not output markdown text explaining your choices; output only the valid tool call.
    """
    # We instantiate a clean agent matching JDExtractionOutput format per run
    local_agent = Agent(
        model,
        model_settings=settings,
        output_type=JDExtractionOutput,
        retries=3
    )
    
    try:
        response = await local_agent.run(dynamic_prompt)
        print(response.output)
        return response.output.model_dump()
    except Exception as e:
        print(f"Extraction failed for title '{row['matched_display_title']}': {e}")
        return None

In [48]:
async def create_groundtruth_dataset(
    df: pd.DataFrame, 
    target_size: int = 20, 
    concurrency_limit: int = 1,  # Set to 1 to process sequentially and avoid 429s
    delay_between_requests: float = 2.0  # Seconds to wait between starting requests
) -> pd.DataFrame:
    """
    Selects 20 diverse, allowed job descriptions from the DataFrame,
    queries the extraction model while limiting concurrency and rate-limits,
    and returns an evaluation dataset.
    """
    # Step 1: Filter and pull exactly 20 diverse records
    sampled_df = extract_and_sample_diverse_jobs(df, target_size=target_size)
    print(f"Selected {len(sampled_df)} diverse jobs across domains.")
    
    # Define a semaphore to throttle concurrent tasks
    semaphore = asyncio.Semaphore(concurrency_limit)
    
    async def throttled_extraction(row, index):
        async with semaphore:
            # Add a staggered delay based on the index to space out execution
            if delay_between_requests > 0 and index > 0:
                await asyncio.sleep(index * delay_between_requests)
                
            print(f"Starting extraction for: {row['matched_display_title']}...")
            result = await run_targeted_extraction(row)
            return result

    # Step 2: Execute throttled extractions
    tasks = [
        throttled_extraction(row, idx) 
        for idx, (_, row) in enumerate(sampled_df.iterrows())
    ]
    extraction_results = await asyncio.gather(*tasks)
    
    # Step 3: Parse extractions and format the evaluation dataframe
    final_rows = []
    for i, (_, row) in enumerate(sampled_df.iterrows()):
        extracted_data = extraction_results[i]
        
        if extracted_data is not None:
            final_rows.append({
                "inputs": {
                    "full_page_text": row["full_page_text"],
                    "url": row["url"]
                },
                "expectations": {
                    "ground_truth_data": extracted_data
                }
            })
            
    print(f"Successfully processed {len(final_rows)} out of {len(sampled_df)} jobs.")
    return pd.DataFrame(final_rows)

In [49]:
# 1. Generate the structured evaluation dataset containing exactly 20 samples
evaluation_df = await create_groundtruth_dataset( df, 
    target_size = 2, 
    concurrency_limit = 1,  
    delay_between_requests= 2.0 
)
# 2. Convert to dictionary objects
eval_rows = evaluation_df.to_dict(orient="records")



Selected 2 diverse jobs across domains.
Starting extraction for: Software Engineer...
job_title='Software Engineer' target_domain='Cloud_Data_and_Connected_Car_Infrastructure' required_toolchains=['GitHub', 'AWS'] compliance_and_standards=['Scrum'] responsibilities=['Coordinate the development, installation, integration and operation of computer-based systems', 'Define system functionality and develop flowcharts, layouts and documentation to identify solutions', 'Develop software solutions by studying systems flow, data usage and work processes', 'Evaluate the performance and reliability of system designs', 'Execute full lifecycle software development', 'Research technical information to design, develop and test computer-based systems', 'Lead and co-ordinate teams of information systems professionals in the development of software, integrated information systems, process control software and embedded software control systems', 'Consult with clients after sale to provide ongoing support

In [50]:
eval_rows

[{'inputs': {'full_page_text': 'software engineer - Mississauga, ON - Job posting - Job Bank\nSkip to job search\nSkip to main content\nSkip to "About this Web application"\nLanguage selection\nFrançais\nfr\nGovernment of Canada /\nGouvernement du Canada\nSearch\nSearch website\nSearch\nJob Bank\nJob Bank\nAccount menu\nSign in\nJob seekers\nEmployers\nMenu and search\nMenu\nMenu\nAccount menu\nSign in\nJob seekers\nEmployers\nMain navigation menu\nJob search\nTraining and careers\nLabour market information\nHiring\nHelp\nAbout\nYou are here:\nJob Bank\nDashboard\nSearch\nJob Search\nKeywords:\nLocation:\nAll of Canada\nCurrent location\nSearch\nAdvanced\nBrowse\nSearch\n21231\n16\nLoading, please wait...\nCancel\nsoftware engineer\nPosted on \n\t\t\t\tMay 19, 2026\nby\nEmployer details\nVisionet Canada Inc\nDirect Apply\nDirect Apply\nSign in to apply directly on Job Bank, or sign up for a Plus account to get started.\nSign up for a Plus account\nSave to favourites\nYour favourites\nT

In [47]:
eval_rows = [{'inputs': {'full_page_text': 'software engineer - Mississauga, ON - Job posting - Job Bank\nSkip to job search\nSkip to main content\nSkip to "About this Web application"\nLanguage selection\nFrançais\nfr\nGovernment of Canada /\nGouvernement du Canada\nSearch\nSearch website\nSearch\nJob Bank\nJob Bank\nAccount menu\nSign in\nJob seekers\nEmployers\nMenu and search\nMenu\nMenu\nAccount menu\nSign in\nJob seekers\nEmployers\nMain navigation menu\nJob search\nTraining and careers\nLabour market information\nHiring\nHelp\nAbout\nYou are here:\nJob Bank\nDashboard\nSearch\nJob Search\nKeywords:\nLocation:\nAll of Canada\nCurrent location\nSearch\nAdvanced\nBrowse\nSearch\n21231\n16\nLoading, please wait...\nCancel\nsoftware engineer\nPosted on \n\t\t\t\tMay 19, 2026\nby\nEmployer details\nVisionet Canada Inc\nDirect Apply\nDirect Apply\nSign in to apply directly on Job Bank, or sign up for a Plus account to get started.\nSign up for a Plus account\nSave to favourites\nYour favourites\nTo add a job posting to your favourites, you need a Job Bank account. Sign in or sign up now!\nSign in\nSign up for a Plus account\nActions\nEmail\nCopy link\nYou have successfully applied for this job through Job Bank!\nYou have successfully withdrawn your application for this job.\nJob details\nEducation: Bachelor\'s degree. Computer science. Computer software engineering. Tasks: Coordinate the development, installation, integration and operation of computer-based systems. Define system functionality. Develop flowcharts, layouts and documentation to identify solutions. Develop software solutions by studying systems flow, data usage and work processes. Evaluate the performance and reliability of system designs. Execute full lifecycle software development. Research technical information to design, develop and test computer-based systems. Lead and co-ordinate teams of information systems professionals in the development of software and integrated information systems, process control software and other embedded software control systems. JavaScript Object Notation (JSON). Robotic process automation. Consult with clients after sale to provide ongoing support. Conduct tests and perform security and quality controls. Computer and technology knowledge: Agile. Development and operations (DevOps). Java. C#. Amazon Web Services (AWS). Go. Python. GitHub. Area of specialization: Scrum. Screening questions: Are you authorized to work in Canada?. Are you willing to relocate for this position?. Do you have experience working in this field?. Do you meet the language requirements listed in the job posting?. Experience: 5 years or more. Health benefits: Dental plan. Vision care benefits. Financial benefits: Registered Retirement Savings Plan (RRSP).\nLocation\n2425 Matheson Blvd E\nMississauga\n,\nON\nL4W 5K4\nWork location\nOn site\nSalary\n$\n57.00\nto\n$\n65.00\nHOUR\nhourly (To be negotiated)\n/\n40 hours per week\nTerms of employment\nPermanent employment\nFull time\nStarts as soon as possible\nBenefits:\nHealth benefits, Financial benefits\nvacancies\n1 vacancy\nSource\nJob Bank\n#3578387\nVarious locations\n2425 Matheson Blvd E\nMississauga, ON\nL4W 5K4\nOverview\nLanguages\nEnglish\nEducation\nBachelor\'s degree\nComputer science\nComputer software engineering\nExperience\n5 years or more\nOn site\nWork must be completed at the physical location. There is no option to work remotely.\nResponsibilities\nTasks\nCoordinate the development, installation, integration and operation of computer-based systems\nDefine system functionality\nDevelop flowcharts, layouts and documentation to identify solutions\nDevelop software solutions by studying systems flow, data usage and work processes\nEvaluate the performance and reliability of system designs\nExecute full lifecycle software development\nResearch technical information to design, develop and test computer-based systems\nLead and co-ordinate teams of information systems professionals in the development of software and integrated information systems, process control software and other embedded software control systems\nJavaScript Object Notation (JSON)\nRobotic process automation\nConsult with clients after sale to provide ongoing support\nConduct tests and perform security and quality controls\nExperience and specialization\nComputer and technology knowledge\nAgile\nDevelopment and operations (DevOps)\nJava\nC#\nAmazon Web Services (AWS)\nGo\nPython\nGitHub\nArea of specialization\nScrum\nBenefits\nHealth benefits\nDental plan\nVision care benefits\nFinancial benefits\nRegistered Retirement Savings Plan (RRSP)\nWho can apply for this job?\nYou can apply if you are:\na Canadian citizen\na permanent resident of Canada\na temporary resident of Canada with a valid work permit\nDo not apply if you are not authorized to work in Canada\n. The employer will not respond to your application.\nShow how to apply\nAdvertised until\n2026-06-08\nImportant notice:\nThis job posting was posted directly by the employer on Job Bank. The Government of Canada has taken steps to make sure it is accurate and reliable but cannot guarantee its authenticity.\nReport a problem with this job posting\n*\nWhat’s wrong?\nThis job posting contains incorrect information\nInaccurate salary\nInaccurate job title\nLink to full job posting / Expired or closed job posting\nEmail\nProvide more details:\nReport potential misuse of Job Bank\nThank you for your help!\nYou will not receive a reply. For enquiries, please\ncontact us\n.\nJob market information\nsoftware engineer\nNOC 21231\nToronto Region\nMedian wage\nHelp -\n56.49 $/hour\nExplore this career\nMedian wage - Help\nThe median wage is the salary of a given occupation where half the workers earn more than that amount, and half earn less. This information is presented on job postings to help job seekers determine how the salary compares to the amount earned by other workers working the same job. Job Bank preferred indicating the median wage, which is less affected by extremely high or low wages, rather than the average wage which is calculated by adding up all the salaries of a group of people and then dividing that total by the number of people.\nClose\nSimilar job postings\n...within Mississauga (ON)\nsoftware engineer\nKloudville Inc.\nsoftware engineer\nSMC Strategic Solutions Inc\nsoftware project manager\nThoughtStorm Incorporated\nsoftware engineer\nSMC Strategic Solutions Inc\nSimilar job postings\nPage details\nReport a problem or mistake on this page\nDate modified:\n2026-04-21\nOpens in a new window\nRelated links\nJob Bank Support\nAbout us\nOur network\nTerms of use - Job seekers\nTerms of use - Employers\nGovernment of Canada Corporate\nTerms and conditions\nThis link will open in a new window\nPrivacy\nThis link will open in a new window\nTop of Page',
   'url': 'https://www.jobbank.gc.ca/jobsearch/jobposting/49543463;jsessionid=35802ED1EF2226C6789927829FA38603.jobsearch76?source=searchresults'},
  'expectations': {'ground_truth_data': {'job_title': 'Software Engineer',
    'target_domain': 'Cloud_Data_and_Connected_Car_Infrastructure',
    'required_toolchains': ['Amazon Web Services (AWS)',
     'GitHub',
     'Java',
     'Go',
     'Python',
     'C#',
     'JSON',
     'DevOps'],
    'compliance_and_standards': ['Agile',
     'Scrum',
     'Security and quality controls',
     'Full lifecycle software development'],
    'responsibilities': ['Coordinate the development, installation, integration and operation of computer-based systems',
     'Define system functionality and develop flowcharts, layouts and documentation to identify solutions',
     'Develop software solutions by studying systems flow, data usage and work processes',
     'Evaluate the performance and reliability of system designs',
     'Lead and co-ordinate teams of information systems professionals in the development of software, integrated information systems, and embedded software control systems',
     'Research technical information to design, develop and test computer-based systems',
     'Conduct tests and perform security and quality controls',
     'Consult with clients after sale to provide ongoing support'],
    'requirements': {'must_have': ["Bachelor's degree in Computer Science or Computer Software Engineering",
      'Proficiency in Java, C#, Go, and Python',
      'Experience with Amazon Web Services (AWS) and GitHub',
      'Authorization to work in Canada',
      'Ability to work on-site in Mississauga, ON'],
     'nice_to_have': ['Experience with Robotic Process Automation (RPA)',
      'Experience with embedded software control systems']},
    'salary_range': '$57.00 - $65.00 per hour',
    'experience': {'years_of_experience': '5 years or more',
     'experience_level': 'Senior'}}}},
 {'inputs': {'full_page_text': 'Web developer - Brampton, ON - Job posting - Job Bank\nSkip to job search\nSkip to main content\nSkip to "About this Web application"\nLanguage selection\nFrançais\nfr\nGovernment of Canada /\nGouvernement du Canada\nSearch\nSearch website\nSearch\nJob Bank\nJob Bank\nAccount menu\nSign in\nJob seekers\nEmployers\nMenu and search\nMenu\nMenu\nAccount menu\nSign in\nJob seekers\nEmployers\nMain navigation menu\nJob search\nTraining and careers\nLabour market information\nHiring\nHelp\nAbout\nYou are here:\nJob Bank\nDashboard\nSearch\nJob Search\nKeywords:\nLocation:\nAll of Canada\nCurrent location\nSearch\nAdvanced\nBrowse\nSearch\n21234\n16\nLoading, please wait...\nCancel\nWeb developer\nPosted on \n\t\t\t\tMarch 30, 2026\nby\na\nlicensed third-party\nfor\nEmployer details\nArora Enterprise Inc\nDirect Apply\nDirect Apply\nSign in to apply directly on Job Bank, or sign up for a Plus account to get started.\nSign up for a Plus account\nSave to favourites\nYour favourites\nTo add a job posting to your favourites, you need a Job Bank account. Sign in or sign up now!\nSign in\nSign up for a Plus account\nActions\nEmail\nCopy link\nYou have successfully applied for this job through Job Bank!\nYou have successfully withdrawn your application for this job.\nJob details\nEducation: College/CEGEP. Tasks: Consult with clients to develop and document Website requirements. Design and integrate website related code. Develop website architecture. Maintain existing computer programs by making modifications as required. Communicate technical problems, processes and solutions. Prepare reports, manuals and other documentation on the status, operation and maintenance of software. Create and optimize content for Website using a variety of graphics, database, animation and other software. Lead and co-ordinate multidisciplinary teams to develop Website graphics, content, capacity and interactivity. Conduct tests and perform security and quality controls. Write, modify, integrate and test software code for e-commerce and other Internet applications. Computer and technology knowledge: Adobe Acrobat Reader. Android. MAC. C++. Programming languages. MS Office. Adobe ActionScript. Adobe XD. Work conditions and physical capabilities: Fast-paced environment. Tight deadlines. Handling heavy loads. Physically demanding. Attention to detail. Sitting. Personal suitability: Accurate. Client focus. Dependability. Efficient interpersonal skills. Excellent oral communication. Initiative. Judgement. Organized. Team player. Integrity. Experience: 7 months to less than 1 year.\nLocation\n10 LIGHTBEAM TERRACE UNIT 19\nBrampton\n,\nON\nL6Y 0B3\nWork location\nOn site\nSalary\n$\n40.00\nHOUR\nhourly\n/\n30 hours per week\nTerms of employment\nTerm or contract\nFull time\nStarts as soon as possible\nvacancies\n1 vacancy\nSource\nJob Bank\n#3540483\nVarious locations\n10 LIGHTBEAM TERRACE UNIT 19\nBrampton, ON\nL6Y 0B3\nOverview\nLanguages\nEnglish\nEducation\nCollege/CEGEP\nExperience\n7 months to less than 1 year\nOn site\nWork must be completed at the physical location. There is no option to work remotely.\nResponsibilities\nTasks\nConsult with clients to develop and document Website requirements\nDesign and integrate website related code\nDevelop website architecture\nMaintain existing computer programs by making modifications as required\nCommunicate technical problems, processes and solutions\nPrepare reports, manuals and other documentation on the status, operation and maintenance of software\nCreate and optimize content for Website using a variety of graphics, database, animation and other software\nLead and co-ordinate multidisciplinary teams to develop Website graphics, content, capacity and interactivity\nConduct tests and perform security and quality controls\nWrite, modify, integrate and test software code for e-commerce and other Internet applications\nExperience and specialization\nComputer and technology knowledge\nAdobe Acrobat Reader\nAndroid\nMAC\nC++\nProgramming languages\nMS Office\nAdobe ActionScript\nAdobe XD\nAdditional information\nWork conditions and physical capabilities\nFast-paced environment\nTight deadlines\nHandling heavy loads\nPhysically demanding\nAttention to detail\nSitting\nPersonal suitability\nAccurate\nClient focus\nDependability\nEfficient interpersonal skills\nExcellent oral communication\nInitiative\nJudgement\nOrganized\nTeam player\nIntegrity\nWho can apply for this job?\nThe employer accepts applications from:\nCanadian citizens and permanent or temporary residents of Canada\nother candidates, with or without a valid Canadian work permit\nShow how to apply\nAdvertised until\n2026-06-01\nImportant notice:\nThis job posting was posted directly by the employer on Job Bank. The Government of Canada has taken steps to make sure it is accurate and reliable but cannot guarantee its authenticity.\nReport a problem with this job posting\n*\nWhat’s wrong?\nThis job posting contains incorrect information\nInaccurate salary\nInaccurate job title\nLink to full job posting / Expired or closed job posting\nEmail\nPhone number\nProvide more details:\nReport potential misuse of Job Bank\nThank you for your help!\nYou will not receive a reply. For enquiries, please\ncontact us\n.\nJob market information\nWeb developer\nNOC 21234\nToronto Region\nMedian wage\nHelp -\n39.42 $/hour\nExplore this career\nMedian wage - Help\nThe median wage is the salary of a given occupation where half the workers earn more than that amount, and half earn less. This information is presented on job postings to help job seekers determine how the salary compares to the amount earned by other workers working the same job. Job Bank preferred indicating the median wage, which is less affected by extremely high or low wages, rather than the average wage which is calculated by adding up all the salaries of a group of people and then dividing that total by the number of people.\nClose\nSimilar job postings\n...within Brampton (ON)\nWeb developer\nJASTEKK IT Consulting\nWeb integrator\nTrans-United Consultants Ltd.\nWeb developer\nVyadom Inc.\nWeb developer\nCanada Diamond Awards\nSimilar job postings\nPage details\nReport a problem or mistake on this page\nDate modified:\n2026-04-21\nOpens in a new window\nRelated links\nJob Bank Support\nAbout us\nOur network\nTerms of use - Job seekers\nTerms of use - Employers\nGovernment of Canada Corporate\nTerms and conditions\nThis link will open in a new window\nPrivacy\nThis link will open in a new window\nTop of Page',
   'url': 'https://www.jobbank.gc.ca/jobsearch/jobposting/49542007;jsessionid=4EE1C4E53400111A8DD642FE90E8F2DE.jobsearch76?source=searchresults'},
  'expectations': {'ground_truth_data': {'job_title': 'Web Developer',
    'target_domain': 'Digital_Product_Companion_Apps_and_UX_UI_Quality',
    'required_toolchains': ['Adobe XD',
     'Adobe ActionScript',
     'Adobe Acrobat Reader',
     'Android',
     'MAC',
     'MS Office',
     'C++'],
    'compliance_and_standards': ['Security and quality controls',
     'E-commerce application standards',
     'Technical documentation and reporting standards'],
    'responsibilities': ['Consult with clients to develop and document Website requirements to align with commercial and user needs',
     'Design, integrate, and test software code for e-commerce and other Internet applications, supporting digital storefront capabilities',
     'Develop website architecture and maintain existing programs through necessary modifications',
     'Create and optimize website content utilizing graphics, databases, and animation software to enhance UX/UI quality',
     'Lead and co-ordinate multidisciplinary teams to develop website graphics, content, capacity, and interactivity',
     'Conduct rigorous tests and perform security and quality controls to ensure site stability and secure communication',
     'Prepare technical reports, manuals, and documentation regarding the status, operation, and maintenance of software',
     'Communicate technical problems and solutions effectively to ensure seamless web infrastructure operations'],
    'requirements': {'must_have': ['College/CEGEP education',
      'Proficiency in programming languages, specifically C++',
      'Ability to design and integrate website related code',
      'Ability to conduct security and quality controls',
      'Client focus and excellent oral communication skills'],
     'nice_to_have': ['Experience with Adobe XD and ActionScript',
      'Knowledge of Android and MAC environments',
      'Experience leading multidisciplinary teams']},
    'salary_range': '$40.00 per hour',
    'experience': {'years_of_experience': '7 months to less than 1 year',
     'experience_level': 'Entry Level'}}}}]

In [51]:

def register_eval_dataset_to_ui(eval_rows, dataset_name):

    exp = mlflow.get_experiment_by_name(exp_name)
    exp_id = exp.experiment_id

    try:
        dataset = create_dataset(name=dataset_name, experiment_id=[exp_id])
    except Exception:
        dataset = get_dataset(name=dataset_name)

    records = []
    for row in eval_rows:
        gt_dict = row["expectations"]
        inputs = row["inputs"]
        records.append({
            "inputs": {
                "full_page_text": inputs["full_page_text"],
                "url": inputs["url"]
            },
            "expectations": {
                "ground_truth_data": gt_dict["ground_truth_data"],
            }
        })

    dataset.merge_records(records)
    print(f"Dataset '{dataset_name}' successfully registered to the Registry.")

In [52]:
register_eval_dataset_to_ui(eval_rows, dataset_name = dataset_job_description)

Dataset 'eval_datset_job_description' successfully registered to the Registry.


# creating CV ground truth dataset 

In [53]:
from schemas import CVExtractionOutput


In [54]:
df_cv = pd.read_csv("cv/Resume/Resume.csv")
df_cv

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR
...,...,...,...,...
2479,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2480,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION
2481,31605080,GEEK SQUAD AGENT Professional...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2482,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION


In [55]:
CV_SYSTEM_PROMPT = """
You are an advanced recruitment intelligence assistant. Your task is to analyze the raw CV/Resume text and extract structured profile parameters matching the target schema.

Please observe these specific extraction rules:
1. CANDIDATE NAME: Extract the full name. If not explicitly found, use the ID context or a fallback placeholder.
2. EDUCATION: Parse academic degrees into structured fields (Degree level, specialization, school, country, year).
3. YEARS OF EXPERIENCE: Calculate total years of active professional work. Treat overlapping dates as continuous (avoid double-counting).
4. CATEGORIZED SKILLS: Carefully classify engineering competencies into our target fields:
   - embedded_firmware: bare-metal C/C++, RTOS, microcontrollers (STM32, Aurix, etc.)
   - high_level_software: Python, Java, Go, JS/TS, etc.
   - vehicle_networks: CAN, CAN-FD, LIN, FlexRay, Automotive Ethernet
   - toolchains_and_validation: Vector CANoe, CANalyzer, dSPACE, debugging tools
   - standards_and_compliance: ISO 26262, AUTOSAR, MISRA, ASPICE
   - cloud_and_telematics: Cloud systems, AWS IoT Core, MQTT, container tools
5. PROJECTS: Extract professional projects/roles chronologically, identifying the tools used and the candidate's contribution.
"""

# Initialize your model as configured previously
cv_extraction_agent = Agent(
    model,
    model_settings=settings,
    system_prompt=CV_SYSTEM_PROMPT,
    output_type=CVExtractionOutput,
    retries=3
)

In [56]:
def sample_diverse_cvs(df_cv: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    """
    Selects N CVs from the source dataset, prioritizing category diversity.
    """
    unique_categories = list(df_cv["Category"].unique())
    sampled_rows = []
    remaining_pool = df_cv.copy()
    
    while len(sampled_rows) < n and not remaining_pool.empty:
        for cat in unique_categories:
            cat_pool = remaining_pool[remaining_pool["Category"] == cat]
            if not cat_pool.empty:
                # Sample 1 record randomly from this category pool
                selected_row = cat_pool.sample(n=1).iloc[0]
                sampled_rows.append(selected_row.to_dict())
                remaining_pool = remaining_pool.drop(selected_row.name)
            if len(sampled_rows) == n:
                break
                
    return pd.DataFrame(sampled_rows)

In [57]:
async def create_cv_groundtruth_dataset(
    df_cv: pd.DataFrame, 
    n: int = 20, 
    concurrency_limit: int = 1, 
    delay_between_requests: float = 2.0
) -> pd.DataFrame:
    """
    Samples N diverse CVs, extracts parameters via the agent with strict rate limits,
    and returns a DataFrame formatted for MLflow registry.
    """
    # Step 1: Extract N diverse candidate resumes
    sampled_df = sample_diverse_cvs(df_cv, n=n)
    print(f"Selected {len(sampled_df)} diverse CVs for extraction.")
    
    # Configure safety semaphore
    semaphore = asyncio.Semaphore(concurrency_limit)
    
    async def throttled_extraction(row: Dict[str, Any], index: int):
        async with semaphore:
            # Space out calls using incremental sleep times
            if delay_between_requests > 0 and index > 0:
                await asyncio.sleep(index * delay_between_requests)
                
            print(f"Extracting profile for Candidate ID: {row['ID']} (Category: {row['Category']})...")
            prompt = f"Candidate ID Context: {row['ID']}\n\nResume Text:\n{row['Resume_str']}"
            
            try:
                response = await cv_extraction_agent.run(prompt)
                return response.output.model_dump()
            except Exception as e:
                print(f"Extraction failed for Candidate ID {row['ID']}: {e}")
                return None

    # Step 2: Execute calls asynchronously under the throttled schema
    tasks = [
        throttled_extraction(row.to_dict(), idx) 
        for idx, (_, row) in enumerate(sampled_df.iterrows())
    ]
    extraction_results = await asyncio.gather(*tasks)
    
    # Step 3: Format outputs to perfectly align with register_eval_dataset_to_ui
    final_rows = []
    for i, (_, row) in enumerate(sampled_df.iterrows()):
        extracted_data = extraction_results[i]
        
        if extracted_data is not None:
            final_rows.append({
                "inputs": {
                    "full_page_text": row["Resume_str"],
                    "url": str(row["ID"])  # Map ID string to url
                },
                "expectations": {
                    "ground_truth_data": extracted_data
                }
            })
            
    print(f"Completed extraction for {len(final_rows)} profiles.")
    return pd.DataFrame(final_rows)

In [58]:
# 1. Run the extraction pipeline for N = 20 CVs
evaluation_cv_df = await create_cv_groundtruth_dataset(df_cv, n=2)

# 2. Extract standard dictionary rows
eval_rows = evaluation_cv_df.to_dict(orient="records")



Selected 2 diverse CVs for extraction.
Extracting profile for Candidate ID: 25150191 (Category: HR)...
Extracting profile for Candidate ID: 85101052 (Category: DESIGNER)...
Completed extraction for 2 profiles.


In [59]:
eval_rows

[{'inputs': {'full_page_text': "         HR CONTACT CENTER SPECIALIST       Summary    Forward-thinking professional with various experience in human resources, sales, customer service and education, offering excellent communication and computer skills; highly organized and meticulous.      Skills          MS Office Suite  Self-motivated professional  Team leadership  Meeting deadlines  Time management skills              Experience     07/2016   to   Current     HR Contact Center Specialist    Company Name   －   City  ,   State      Answer and resolve employee and people-manager issues including navigational support and processing corrective transactions when required.  Provide advice on how to complete requests and/or options for next steps based on scenarios.  These could include; guidance related to completing HR responsibilities (year- end compensation, mid-year and year-end processes, resource planning), guidance related to making employee data changes (new hires, transfers, term

In [60]:
# 3. Register directly to your MLflow instance using your existing helper
register_eval_dataset_to_ui(eval_rows, dataset_name = dataset_CV_description)

Dataset 'eval_datset_CV_description' successfully registered to the Registry.


# Create the matcher ground truth dataset for evaluation

In [59]:
from schemas import MatchOutput

MATCHER_SYSTEM_PROMPT = """
You are a highly precise candidate-to-job matching agent. You must output your response STRICTLY as a tool call conforming to the expected JSON schema.

Do not write conversational text, markdown intros, or summary reports. Go straight to returning the structured JSON.

--- Expected Output JSON Format ---
{
  "score_breakdown": {
    "must_have": {
      "weight": 40,
      "score": <int>,
      "matched_items": [<string>, ...],
      "missing_items": [<string>, ...],
      "justification": <string>
    },
    "experience": {
      "weight": 25,
      "score": <int>,
      "matched_items": [<string>, ...],
      "missing_items": [<string>, ...],
      "justification": <string>
    },
    "domain": {
      "weight": 15,
      "score": <int>,
      "matched_items": [<string>, ...],
      "missing_items": [<string>, ...],
      "justification": <string>
    },
    "toolchain": {
      "weight": 10,
      "score": <int>,
      "matched_items": [<string>, ...],
      "missing_items": [<string>, ...],
      "justification": <string>
    },
    "nice_to_have": {
      "weight": 5,
      "score": <int>,
      "matched_items": [<string>, ...],
      "missing_items": [<string>, ...],
      "justification": <string>
    },
    "standards": {
      "weight": 5,
      "score": <int>,
      "matched_items": [<string>, ...],
      "missing_items": [<string>, ...],
      "justification": <string>
    },
    "responsibilities": {
      "weight": 5,
      "score": <int>,
      "matched_items": [<string>, ...],
      "missing_items": [<string>, ...],
      "justification": <string>
    }
  },
  "matched_skills": [<string>, ...],
  "missing_critical_skills": [<string>, ...],
  "missing_soft_skills": [<string>, ...]
}

CRITICAL RULES:
1. Every category inside 'score_breakdown' (must_have, experience, domain, toolchain, nice_to_have, standards, responsibilities) must be a structured object with 'weight', 'score', 'matched_items', 'missing_items', and 'justification'. Do NOT output them as plain integers.
2. If any list has no items (like missing_soft_skills or missing_items), populate it as an empty list: [].
"""

matcher_agent = Agent(
    model,
    model_settings=settings,
    system_prompt=MATCHER_SYSTEM_PROMPT,
    output_type=MatchOutput,
    retries=3
)

In [60]:
async def create_matcher_groundtruth_dataset(
    cv_extractions: List[Dict[str, Any]], 
    jd_extractions: List[Dict[str, Any]],
    concurrency_limit: int = 1,
    delay_between_requests: float = 2.0
) -> pd.DataFrame:
    """
    Pairs CV and JD extractions, executes structured matching,
    and returns a DataFrame matching the target MLflow evaluation schema.
    """
    pair_count = min(len(cv_extractions), len(jd_extractions))
    print(f"Generating matching evaluations for {pair_count} pairs.")
    
    semaphore = asyncio.Semaphore(concurrency_limit)
    
    async def throttled_match(cv_data: Dict[str, Any], jd_data: Dict[str, Any], index: int) -> Optional[Dict[str, Any]]:
        async with semaphore:
            if delay_between_requests > 0 and index > 0:
                await asyncio.sleep(index * delay_between_requests)
                
            print(f"Evaluating Match {index + 1}/{pair_count} (Candidate: {cv_data.get('candidate_name', 'Unknown')} <-> Role: {jd_data.get('job_title', 'Unknown')})...")
            
            # Pack inputs matching the MatchInput schema
            match_input = {
                "cv_data": cv_data,
                "jd_data": jd_data
            }
            
            prompt = f"Perform alignment on the following payload:\n{json.dumps(match_input)}"
            
            try:
                response = await matcher_agent.run(prompt)
                return response.output.model_dump()
            except Exception as e:
                print(f"Matcher failed on index {index}: {e}")
                return None

    # Execute pairing queries sequentially
    tasks = [
        throttled_match(cv_extractions[i], jd_extractions[i], i) 
        for i in range(pair_count)
    ]
    matching_results = await asyncio.gather(*tasks)
    
    # Format database rows to align with register_eval_dataset_to_ui expectation
    final_rows = []
    for i in range(pair_count):
        match_result = matching_results[i]
        
        if match_result is not None:
            cv_item = cv_extractions[i]
            jd_item = jd_extractions[i]
            
            # Serialized JSON containing structured inputs for evaluation references
            combined_input_str = json.dumps({
                "cv_data": cv_item,
                "jd_data": jd_item
            })
            
            final_rows.append({
                "inputs": {
                    "full_page_text": combined_input_str,
                    "url": f"match_cv_{i}_jd_{i}"  # Logical unique identifier matching url structure
                },
                "expectations": {
                    "ground_truth_data": match_result
                }
            })
            
    print(f"Successfully generated {len(final_rows)} match records.")
    return pd.DataFrame(final_rows)

In [61]:
def load_eval_rows_from_registry(dataset_name):
    import json
    from mlflow.genai.datasets import get_dataset
    
    print(f"Fetching records for dataset: {dataset_name}...")
    dataset = get_dataset(dataset_name)
    df = dataset.to_df()
    
    reconstructed_eval_rows = []
    
    for _, row in df.iterrows():
        inputs_dict = row['inputs']
        expectations_dict = row['expectations']
        
        # Extract the ground truth dict we nested earlier
        gt_dict = expectations_dict.get("ground_truth_data", {})
        
        reconstructed_eval_rows.append({
            "filename": inputs_dict.get("url"),
            "article_text": inputs_dict.get("full_page_text"),
            # Convert back to JSON string to match your pipeline's expectation
            "ground_truth": json.dumps(gt_dict) 
        })
        
    print(f"Successfully reconstructed {len(reconstructed_eval_rows)} rows.")
    return reconstructed_eval_rows

In [9]:
def safe_parse_json(val):
    if isinstance(val, str):
        try:
            return json.loads(val)
        except json.JSONDecodeError:
            return val
    return val

In [31]:
job_eval_data = load_eval_rows_from_registry(dataset_name=dataset_job_description)
cv_eval_data = load_eval_rows_from_registry(dataset_name=dataset_CV_description) 
job_eval_data = safe_parse_json(job_eval_data)
cv_eval_data = safe_parse_json(cv_eval_data)

Fetching records for dataset: eval_datset_job_description...
Successfully reconstructed 2 rows.
Fetching records for dataset: eval_datset_CV_description...
Successfully reconstructed 2 rows.


In [29]:
job_eval_data

[{'filename': None,
  'article_text': None,
  'ground_truth': '{"job_title": "Web developer", "target_domain": "Digital_Product_Companion_Apps_and_UX_UI_Quality", "required_toolchains": ["Adobe XD", "Adobe ActionScript", "C++", "Android", "MAC", "MS Office", "Adobe Acrobat Reader"], "compliance_and_standards": [], "responsibilities": ["Consult with clients to develop and document Website requirements", "Design and integrate website related code", "Develop website architecture", "Maintain existing computer programs by making modifications as required", "Communicate technical problems, processes and solutions", "Prepare reports, manuals and other documentation on the status, operation and maintenance of software", "Create and optimize content for Website using a variety of graphics, database, animation and other software", "Lead and co-ordinate multidisciplinary teams to develop Website graphics, content, capacity and interactivity", "Conduct tests and perform security and quality contro

In [75]:
# 1. Retrieve lists of structured dicts extracted from your prior runs
cv_dict_list = [safe_parse_json(row["ground_truth"]) for row in cv_eval_data]
jd_dict_list = [safe_parse_json(row["ground_truth"]) for row in job_eval_data]

# 2. Run the matcher pipeline (paired sequentially up to N = 20)
matcher_evaluation_df = await create_matcher_groundtruth_dataset(
    cv_extractions=cv_dict_list,
    jd_extractions=jd_dict_list,
    concurrency_limit=1,
    delay_between_requests=2.0
)

# 3. Convert generated dataframe to records
matcher_eval_rows = matcher_evaluation_df.to_dict(orient="records")



Generating matching evaluations for 2 pairs.
Evaluating Match 1/2 (Candidate: E. April Bradford <-> Role: Web developer)...
Evaluating Match 2/2 (Candidate: Not Explicitly Provided <-> Role: Software Engineer)...
Successfully generated 2 match records.


In [76]:
matcher_eval_rows

[{'inputs': {'full_page_text': '{"cv_data": {"candidate_name": "E. April Bradford", "education": [{"degree_level": "MBA", "field_of_study": null, "institution": null, "country": "USA", "graduation_year": null}, {"degree_level": "Bachelor of Arts", "field_of_study": null, "institution": null, "country": "USA", "graduation_year": null}], "years_of_experience": 10.0, "categorized_skills": {"embedded_firmware": [], "high_level_software": [], "vehicle_networks": [], "toolchains_and_validation": [], "standards_and_compliance": [], "cloud_and_telematics": []}, "projects": [{"project_name": "HR Contact Center Operations", "duration_months": null, "tools_used": ["MS Office Suite", "HR Systems"], "contribution": "Resolving employee/manager issues, providing guidance on HR responsibilities (compensation, resource planning), managing employee data changes, and ensuring audit/compliance adherence. Involved in new hire training and women in leadership initiatives."}, {"project_name": "Student Admiss

In [77]:
# 4. Register using your helper script to MLflow
register_eval_dataset_to_ui(matcher_eval_rows, dataset_name=dataset_Matcher_description)

Dataset 'eval_datset_Matcher_description' successfully registered to the Registry.


In [32]:

matcher_eval_data = load_eval_rows_from_registry(dataset_name=dataset_Matcher_description)
matcher_eval_data = safe_parse_json(matcher_eval_data)

Fetching records for dataset: eval_datset_Matcher_description...
Successfully reconstructed 2 rows.


# Extractor and judge

In [18]:
from schemas import (FieldVerdict, 
                     JDVerdict,
                     JDJudgeResponse,
                      CVVerdict,
                      CVJudgeResponse,
                      MatcherVerdict,
                    MatcherJudgeResponse,
                    JDExtractionOutput,
                    CVExtractionOutput,
                     MatchOutput,
                    JDJudgeResponse)


In [62]:
SCHEMA_CONSTRAINTS = """
--- SCHEMA CONSTRAINTS ---
For the JSON output, each field in the 'verdicts' object must conform strictly to the FieldVerdict schema:
- Use 'is_correct' (do not use 'status').
- Use 'explanation' (do not use 'comment' or 'rationale').
"""

JD_JUDGE_PROMPT = """
You are a strict data auditor. Your job is to compare 'Extracted' Job Description (JD) data against 'Ground Truth' for specific fields.

CONTEXT:
JD Text: {{ inputs }}

DATA TO EVALUATE:
Prediction: {{ outputs }}
Ground Truth: {{ expectations }}

TASK:
Determine whether each extracted field is strictly truthful according to the reference JD text and corporate guidelines.
Compare 'Extracted' against 'Ground Truth'.
- Assign 1 (Correct) if they are semantically identical.
- Assign 0 (Incorrect) if there is any factual contradiction, missing detail, or hallucination.
Do not allow partial matches, paraphrases that change intent, or approximations.

""" + SCHEMA_CONSTRAINTS

CV_JUDGE_PROMPT = """
You are a strict data auditor. Your job is to compare 'Extracted' CV data against 'Ground Truth' for specific fields.

CONTEXT:
CV Text: {{ inputs }}

DATA TO EVALUATE:
Prediction: {{ outputs }}
Ground Truth: {{ expectations }}

TASK:
Determine whether each extracted CV field is strictly truthful according to the reference CV text.
Compare 'Extracted' against 'Ground Truth'.
- Assign 1 (Correct) if they are semantically identical.
- Assign 0 (Incorrect) if there is a factual contradiction or missing information.
Calculate years of professional experience with absolute precision.

""" + SCHEMA_CONSTRAINTS

MATCHER_JUDGE_PROMPT = """
You are a strict auditing system checking Candidate-to-Job matching evaluations.

CONTEXT:
Evaluated Profiles (CV + JD payload): {{ inputs }}

DATA TO EVALUATE:
Prediction: {{ outputs }}
Ground Truth: {{ expectations }}

TASK:
Verify whether the matcher successfully aligned and calculated alignment points according to the target guidelines.
Assess whether:
1. The score breakdown correctly matched criteria.
2. The extracted arrays (matched skills, missing critical/soft skills) correspond perfectly to the target profile mapping.
- Assign 1 (Correct) if they are semantically aligned and verified.
- Assign 0 (Incorrect) if there is any point calculation error or mismatched parameter.

""" + SCHEMA_CONSTRAINTS

In [49]:
jd_extraction_agent = Agent(
    model,
    model_settings=settings,
    system_prompt="Extract structured attributes from the Job Description text in accordance with our taxonomy.",
    output_type=JDExtractionOutput,
    retries=3
)

cv_extraction_agent = Agent(
    model,
    model_settings=settings,
    system_prompt="Extract structured details from the candidate CV text in accordance with our taxonomy.",
    output_type=CVExtractionOutput,
    retries=3
)

matcher_extraction_agent = Agent(
    model,
    model_settings=settings,
    system_prompt="Perform matching and scoring calculations for the given candidate profile against the JD requirements.",
    output_type=MatchOutput,
    retries=3
)

# --- Judge Agents (Auditing) ---
jd_judge_agent = Agent(
    model,
    model_settings=settings,
    system_prompt=JD_JUDGE_PROMPT,
    output_type=JDJudgeResponse,
    retries=3
)

cv_judge_agent = Agent(
    model,
    model_settings=settings,
    system_prompt=CV_JUDGE_PROMPT,
    output_type=CVJudgeResponse,
    retries=3
)

matcher_judge_agent = Agent(
    model,
    model_settings=settings,
    system_prompt=MATCHER_JUDGE_PROMPT,
    output_type=MatcherJudgeResponse,
    retries=3
)

In [50]:
# Thread-safe global references for execution binding
active_jd_judge_agent = jd_judge_agent
active_cv_judge_agent = cv_judge_agent
active_matcher_judge_agent = matcher_judge_agent

@mlflow.genai.scorer
def jd_truthfulness_scorer(inputs, outputs, expectations, **kwargs):
    global active_jd_judge_agent
    prompt = f"""
    Evaluate these specific JD fields:
    GROUND TRUTH: {expectations.get('targets')}
    EXTRACTED: {outputs}
    REFERENCE TEXT: {inputs.get('article_text')}
    """
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        result = loop.run_until_complete(active_jd_judge_agent.run(prompt))
        return Feedback(
            name="jd_truthfulness_score",
            value=result.output.score,      
            rationale=result.output.model_dump_json()
        )
    finally:
        loop.close()

@mlflow.genai.scorer
def cv_truthfulness_scorer(inputs, outputs, expectations, **kwargs):
    global active_cv_judge_agent
    prompt = f"""
    Evaluate these specific CV fields:
    GROUND TRUTH: {expectations.get('targets')}
    EXTRACTED: {outputs}
    REFERENCE TEXT: {inputs.get('article_text')}
    """
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        result = loop.run_until_complete(active_cv_judge_agent.run(prompt))
        return Feedback(
            name="cv_truthfulness_score",
            value=result.output.score,      
            rationale=result.output.model_dump_json()
        )
    finally:
        loop.close()

@mlflow.genai.scorer
def matcher_truthfulness_scorer(inputs, outputs, expectations, **kwargs):
    global active_matcher_judge_agent
    prompt = f"""
    Evaluate these specific Matcher fields:
    GROUND TRUTH: {expectations.get('targets')}
    EXTRACTED: {outputs}
    REFERENCE TEXT: {inputs.get('article_text')}
    """
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        result = loop.run_until_complete(active_matcher_judge_agent.run(prompt))
        return Feedback(
            name="matcher_truthfulness_score",
            value=result.output.score,      
            rationale=result.output.model_dump_json()
        )
    finally:
        loop.close()

In [63]:
# Exponential retry wrapper for processing calls
def retry_async(max_retries=4, initial_delay=2.0, backoff_factor=2.0, jitter=True):
    def decorator(func):
        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            delay = initial_delay
            for attempt in range(1, max_retries + 2):
                try:
                    return await func(*args, **kwargs)
                except Exception as e:
                    if attempt > max_retries:
                        print(f" Permanent failure for {func.__name__} after {max_retries} retries: {e}")
                        raise e
                    sleep_time = delay * random.uniform(0.5, 1.5) if jitter else delay
                    print(f" {func.__name__} failed (Attempt {attempt}/{max_retries + 1}): {e}. Retrying in {sleep_time:.2f} seconds...")
                    await asyncio.sleep(sleep_time)
                    delay *= backoff_factor
        return wrapper
    return decorator


@retry_async(max_retries=4, initial_delay=2.0)
async def execute_agent_extraction(agent: Agent, text_payload: str) -> Dict[str, Any]:
    """Runs a target extraction agent against the text payload with verbose error capture."""
    with capture_run_messages() as messages:
        try:
            run_result = await agent.run(text_payload)
            return run_result.output.model_dump()
        except Exception as e:
            print(f"\n Extraction failed for agent: {e}")
            print("--- AGENT MESSAGES & VALIDATION TRACE ---")
            for msg in messages:
                # This will display the validation errors Pydantic sent to the LLM
                print(msg)
            print("-------------------------------------------\n")
            raise e

# Concurrency semaphore
MAX_PARALLEL_TASKS = 4
pipeline_semaphore = asyncio.Semaphore(MAX_PARALLEL_TASKS)

async def process_single_pipeline_document(
    item: Dict[str, Any], 
    extraction_agent: Agent, 
    scorer_function: Any, 
    scorer_name: str, 
    version_label: str
) -> Dict[str, Any]:
    """
    Encapsulates extraction, custom MLflow evaluation, and 
    automated retries for a single document on a generic pipeline.
    """
    # Extract structural inputs
    article_text = item["article_text"]
    filename = item.get("filename", item.get("filename", "unknown"))
    ground_truth = item["ground_truth"]

    if isinstance(ground_truth, str):
        ground_truth = json.loads(ground_truth)

    async with pipeline_semaphore:
        print(f" Starting extraction/evaluation on: {filename}")
        
        # 1. Extraction (Parallel with retry logic)
        output_dict = await execute_agent_extraction(extraction_agent, article_text)

        # 2. Package payloads for MLflow evaluate compatibility
        single_doc_eval_data = [{
            "inputs": {"article_text": article_text},
            "outputs": output_dict,
            "targets": ground_truth,
            "expectations": {"targets": ground_truth} 
        }]

        per_field_verdict = {}
        current_trace_id = None
        current_run_id = None

        # 3. Evaluation with manual exponential backoff retry loop
        max_mlflow_retries = 4
        mlflow_delay = 2.0
        
        for attempt in range(1, max_mlflow_retries + 2):
            try:
                with mlflow.start_run(run_name=f"J{version_label}_{filename}", nested=True) as run:
                    eval_report = mlflow.genai.evaluate(
                        data=single_doc_eval_data, 
                        scorers=[scorer_function] 
                    )
                    
                    df = eval_report.tables["eval_results"]
                    current_trace_id = df["trace_id"].iloc[0]
                    current_run_id = run.info.run_id
                    
                    if "assessments" in df.columns and not df["assessments"].empty:
                        row_assessments = df["assessments"].iloc[0]
                        for assessment in row_assessments:
                            if assessment.get("assessment_name") == scorer_name:
                                raw_rationale = assessment.get("rationale")
                                if raw_rationale:
                                    parsed = json.loads(raw_rationale)
                                    per_field_verdict = parsed.get("verdicts", {})
                                    break
                break  # Exit retry loop on success

            except Exception as e:
                if attempt > max_mlflow_retries:
                    print(f" MLflow evaluation permanent failure on {filename} after {max_mlflow_retries} retries: {e}")
                    break
                
                sleep_time = mlflow_delay * random.uniform(0.5, 1.5)
                print(f" MLflow evaluate failed for {filename} (Attempt {attempt}/{max_mlflow_retries + 1}): {e}. Retrying in {sleep_time:.2f} seconds...")
                await asyncio.sleep(sleep_time)
                mlflow_delay *= 2.0

        print(f" Finished extraction/evaluation on: {filename}")
        
        return {
            "filename": filename,
            "trace_id": current_trace_id, 
            "run_id": current_run_id,
            "extracted": output_dict,
            "truth": ground_truth,
            "verdicts": per_field_verdict 
        }

async def run_evaluation_pipeline_parallel(
    eval_rows: List[Dict[str, Any]], 
    extraction_agent: Agent,
    scorer_function: Any,
    scorer_name: str,
    version_label: str = "v1"
) -> List[Dict[str, Any]]:
    """Runs parallelized evaluation pipeline for any target configuration."""
    tasks = [
        process_single_pipeline_document(
            item, 
            extraction_agent, 
            scorer_function, 
            scorer_name, 
            version_label
        ) 
        for item in eval_rows
    ]
    results = await asyncio.gather(*tasks)
    return list(results)

In [ ]:
# --- Run JD Extraction & Verification Pipeline ---
global active_jd_judge_agent
active_jd_judge_agent = jd_judge_agent

jd_pipeline_results = await run_evaluation_pipeline_parallel(
    eval_rows=job_eval_data,             # Generated during JD step
    extraction_agent=jd_extraction_agent,
    scorer_function=jd_truthfulness_scorer,
    scorer_name="jd_truthfulness_score",
    version_label="jd_v1"
)

# --- Run CV Extraction & Verification Pipeline ---
global active_cv_judge_agent
active_cv_judge_agent = cv_judge_agent

cv_pipeline_results = await run_evaluation_pipeline_parallel(
    eval_rows=cv_eval_data,             # Generated during CV step
    extraction_agent=cv_extraction_agent,
    scorer_function=cv_truthfulness_scorer,
    scorer_name="cv_truthfulness_score",
    version_label="cv_v1"
)


 Starting extraction/evaluation on: https://www.jobbank.gc.ca/jobsearch/jobposting/49542007;jsessionid=4EE1C4E53400111A8DD642FE90E8F2DE.jobsearch76?source=searchresults
 Starting extraction/evaluation on: https://www.jobbank.gc.ca/jobsearch/jobposting/49543463;jsessionid=35802ED1EF2226C6789927829FA38603.jobsearch76?source=searchresults


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

 Finished extraction/evaluation on: https://www.jobbank.gc.ca/jobsearch/jobposting/49543463;jsessionid=35802ED1EF2226C6789927829FA38603.jobsearch76?source=searchresults
 execute_agent_extraction failed (Attempt 1/5): Exceeded maximum output retries (3). Retrying in 2.35 seconds...


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

 Finished extraction/evaluation on: https://www.jobbank.gc.ca/jobsearch/jobposting/49542007;jsessionid=4EE1C4E53400111A8DD642FE90E8F2DE.jobsearch76?source=searchresults
 Starting extraction/evaluation on: 25150191
 Starting extraction/evaluation on: 85101052


...

 Finished extraction/evaluation on: 85101052


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

 Finished extraction/evaluation on: 25150191
 Starting extraction/evaluation on: match_cv_1_jd_1
 Starting extraction/evaluation on: match_cv_0_jd_0
 execute_agent_extraction failed (Attempt 1/5): Exceeded maximum output retries (3). Retrying in 2.74 seconds...
 execute_agent_extraction failed (Attempt 1/5): Exceeded maximum output retries (3). Retrying in 1.55 seconds...
 execute_agent_extraction failed (Attempt 2/5): Exceeded maximum output retries (3). Retrying in 3.22 seconds...
 execute_agent_extraction failed (Attempt 2/5): Exceeded maximum output retries (3). Retrying in 5.75 seconds...
 execute_agent_extraction failed (Attempt 3/5): Exceeded maximum output retries (3). Retrying in 4.90 seconds...
 execute_agent_extraction failed (Attempt 3/5): Exceeded maximum output retries (3). Retrying in 8.37 seconds...
 execute_agent_extraction failed (Attempt 4/5): Exceeded maximum output retries (3). Retrying in 20.48 seconds...
 execute_agent_extraction failed (Attempt 4/5): Exceeded ma

[Trace(trace_id=tr-37092aa2361cc4c5a81e218135880110), Trace(trace_id=tr-0582394425302f87342ebeee884f41ce), Trace(trace_id=tr-93174d59b6df9edbe980ad72e9a3561b), Trace(trace_id=tr-4c751faad62ddee6b87b43734fa03bde)]

 Permanent failure for execute_agent_extraction after 4 retries: Exceeded maximum output retries (3)


In [64]:

# --- Run Matcher & Verification Pipeline ---
global active_matcher_judge_agent
active_matcher_judge_agent = matcher_judge_agent

matcher_pipeline_results = await run_evaluation_pipeline_parallel(
    eval_rows=matcher_eval_data,        # Generated during Matcher step
    extraction_agent=matcher_extraction_agent,
    scorer_function=matcher_truthfulness_scorer,
    scorer_name="matcher_truthfulness_score",
    version_label="matcher_v1"
)

 Starting extraction/evaluation on: match_cv_1_jd_1
 Starting extraction/evaluation on: match_cv_0_jd_0


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

 Finished extraction/evaluation on: match_cv_0_jd_0


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

 Finished extraction/evaluation on: match_cv_1_jd_1


Trace(trace_id=tr-304767ba620734f477657690d8d80951)

In [65]:
matcher_pipeline_results

[{'filename': 'match_cv_1_jd_1',
  'trace_id': 'tr-bb7220ecbaf6b75ce1f0dc4fd2aea844',
  'run_id': '92a07c5ac5e24496ae7c48aa24e99a60',
  'extracted': {'score_breakdown': {'must_have': {'weight': 40,
     'score': 15,
     'matched_items': ['Java',
      'Full lifecycle software development experience'],
     'missing_items': ['Go',
      'Python',
      'AWS (Amazon Web Services)',
      'Agile',
      'DevOps'],
     'justification': 'The candidate is a strong Java developer but lacks critical modern backend languages (Go, Python) and essential cloud/infrastructure skills (AWS, DevOps) required for this specific role.'},
    'experience': {'weight': 25,
     'score': 25,
     'matched_items': ['5 years or more experience'],
     'missing_items': [],
     'justification': 'With 17 years of experience, the candidate far exceeds the minimum requirement of 5 years.'},
    'domain': {'weight': 15,
     'score': 0,
     'matched_items': [],
     'missing_items': ['Cloud_Data_and_Connected_Ca

# Human feedback data collection analysis to judge report

In [ ]:
JD_FIELDS = [
    "job_title", 
    "target_domain", 
    "required_toolchains", 
    "compliance_and_standards", 
    "responsibilities", 
    "requirements", 
    "salary_range", 
    "experience"
]

CV_FIELDS = [
    "candidate_name", 
    "education", 
    "years_of_experience", 
    "categorized_skills", 
    "projects"
]

MATCHER_FIELDS = [
    "score_breakdown", 
    "matched_skills", 
    "missing_critical_skills", 
    "missing_soft_skills"
]

In [ ]:
def display_comparison_table(reports: List[Dict[str, Any]], fields: List[str]) -> pd.DataFrame:
    """
    Constructs a flattened pandas DataFrame comparing extracted values 
    against ground truth and displaying judge verdicts for any pipeline.
    """
    flat_data = []

    for item in reports:
        fname = item.get("filename", "unknown")
        ext_dict = item.get("extracted", {})
        tru_dict = item.get("truth", {})
        verdicts = item.get("verdicts", {})

        for field in fields:
            field_verdict = verdicts.get(field)
            
            if field_verdict:
                raw_score = field_verdict.get("is_correct", 0)
                is_correct = (raw_score == 1 or raw_score is True)
                reasoning = field_verdict.get("explanation", "No explanation provided.")
            else:
                is_correct = False
                reasoning = "⚠️ Judge missed this field or extraction failed."

            flat_data.append({
                "Document": fname,
                "Field": field.replace("_", " ").title(),
                "Agent Extracted": str(ext_dict.get(field, "NOT EXTRACTED")),
                "Ground Truth": str(tru_dict.get(field, "MISSING IN GT")),
                "Match": "No" if is_correct else "Yes",
                "Judge Reasoning": reasoning
            })

    if not flat_data:
        raise ValueError("No comparison records were compiled.")

    df = pd.DataFrame(flat_data)
    df = df.set_index(["Document", "Field"])
    return df

In [ ]:
comparison_df_jd = display_comparison_table(jd_pipeline_results, JD_FIELDS)
comparison_df_cv = display_comparison_table(jd_pipeline_results, CV_FIELDS)
comparison_df_matcher = display_comparison_table(jd_pipeline_results, MATCHER_FIELDS)



In [ ]:
comparison_df_jd

In [ ]:
comparison_df_cv

In [ ]:
comparison_df_matcher

In [ ]:
jd_human_feedback = {
    "human_feedback_data": [
        # --- Document 1 (Must contain exactly 8 elements in this order) ---
        {"human_score": 1, "human_feedback": "Correct. Matches title 'Systems Engineer' perfectly."},
        {"human_score": 1, "human_feedback": "Correctly resolved to the Vehicle Tech domain."},
        {"human_score": 1, "human_feedback": "Accurately captured CANoe and Simulink from requirements."},
        {"human_score": 0, "human_feedback": "Incorrect. The judge missed verifying ISO 26262 in compliance."},
        {"human_score": 1, "human_feedback": "Fidelity of extracted responsibilities matches text."},
        {"human_score": 1, "human_feedback": "Must-have and Nice-to-have parameters match text."},
        {"human_score": 1, "human_feedback": "Accurately marked salary as 'Not Specified'."},
        {"human_score": 1, "human_feedback": "Correctly extracted 5 years of required experience."},
        
        # --- Document 2 (Repeat 8 elements for Document 2, etc.) ---
    ]
}

cv_human_feedback = {
    "human_feedback_data": [
        # --- Document 1 (Must contain exactly 5 elements in this order) ---
        {"human_score": 1, "human_feedback": "Captured 'Jane Doe' accurately."},
        {"human_score": 1, "human_feedback": "Accurately extracted Master's in Robotics and University context."},
        {"human_score": 1, "human_feedback": "Parsed active employment dates to calculate 4.5 years experience."},
        {"human_score": 1, "human_feedback": "Competently sorted STM32 and ROS into respective firmware/software arrays."},
        {"human_score": 0, "human_feedback": "Incomplete. Missing the 'Autonomous Shuttles' project details."},
        
        # --- Document 2 (Repeat 5 elements for Document 2, etc.) ---
    ]
}

matcher_human_feedback = {
    "human_feedback_data": [
        # --- Document 1 (Must contain exactly 4 elements in this order) ---
        {"human_score": 1, "human_feedback": "The points math is correct and reflects the criteria rules."},
        {"human_score": 1, "human_feedback": "All overlapping skills and experience parameters matched cleanly."},
        {"human_score": 1, "human_feedback": "Accurately flagged missing ISO 26262 competency as a critical deficiency."},
        {"human_score": 1, "human_feedback": "Nice-to-have gaps correctly captured in soft skills array."},
        
        # --- Document 2 (Repeat 4 elements for Document 2, etc.) ---
    ]
}

In [ ]:
async def upload_human_feedback(
    results: List[Dict[str, Any]], 
    feedback_payload: Dict[str, Any], 
    fields: List[str]
):
    """
    Uploads flattened human feedback data to MLflow, chunking items 
    dynamically matching the number of fields in the active pipeline.
    """
    all_feedback_items = feedback_payload["human_feedback_data"]
    chunk_size = len(fields)
    
    # Chunk the flat feedback list into groups of size `chunk_size` (one group per document)
    feedback_chunks = [
        all_feedback_items[i:i + chunk_size] 
        for i in range(0, len(all_feedback_items), chunk_size)
    ]

    for doc_idx, doc_data in enumerate(results):
        trace_id = doc_data.get("trace_id")
        filename = doc_data.get("filename")
        
        if not trace_id:
            print(f" Skipping feedback upload for {filename}: No trace_id found.")
            continue
            
        current_doc_feedback_chunk = feedback_chunks[doc_idx]
        print(f"Uploading {chunk_size} human feedback items for {filename} (Trace: {trace_id})...")

        for field_idx, field_name in enumerate(fields):
            feedback_item = current_doc_feedback_chunk[field_idx]
            
            mlflow.log_feedback(
                trace_id=trace_id,
                name=f"Human_{field_name}",
                value=float(feedback_item["human_score"]),
                rationale=feedback_item["human_feedback"]
            )

    print("\n Structured human feedback successfully registered to MLflow UI.")

In [ ]:
await upload_human_feedback(comparison_df_jd, jd_human_feedback, JD_FIELDS)
await upload_human_feedback(comparison_df_cv, cv_human_feedback, CV_FIELDS)
await upload_human_feedback(comparison_df_matcher, matcher_human_feedback, MATCHER_FIELDS)


In [ ]:
def analyze_alignment(results: List[Dict[str, Any]], human_payload: Dict[str, Any], fields: List[str]) -> pd.DataFrame:
    """
    Compares AI judge verdicts against human-assigned scores for convergence mapping.
    """
    human_list = human_payload["human_feedback_data"]
    chunk_size = len(fields)
    alignment_data = []

    for doc_idx, doc_data in enumerate(results):
        judge_verdicts = doc_data.get("verdicts", {})
        # Isolate the segment matching this specific document
        human_chunk = human_list[doc_idx * chunk_size : (doc_idx + 1) * chunk_size]

        for field_idx, field in enumerate(fields):
            field_v = judge_verdicts.get(field)
            # Evaluate judge score as integer (correct = 1, incorrect = 0)
            if field_v:
                raw_judge = field_v.get("is_correct", 0)
                judge_score = 1 if (raw_judge == 1 or raw_judge is True) else 0
            else:
                judge_score = 0
            
            human_score = int(human_chunk[field_idx]["human_score"])
            status = "Convergent (Agreement)" if judge_score == human_score else "Divergent (Disagreement)"
            
            alignment_data.append({
                "Field": field.replace("_", " ").title(),
                "Status": status
            })

    return pd.DataFrame(alignment_data)

In [ ]:
jd_alignment = analyze_alignment(jd_pipeline_results, jd_human_feedback, JD_FIELDS)
cv_alignment = analyze_alignment(jd_pipeline_results, cv_human_feedback, CV_FIELDS)
matcher_alignment = analyze_alignment(jd_pipeline_results, matcher_human_feedback, MATCHER_FIELDS)



In [ ]:

def plot_convergence(df: pd.DataFrame, title_label: str):
    """
    Plots stacked alignment distribution across fields for the selected pipeline.
    """
    plot_df = df.groupby(['Field', 'Status']).size().unstack(fill_value=0)
    
    # Classic UI reporting color theme (Green = Agreement, Red = Disagreement)
    colors = ["#2ecc71", "#e74c3c"]
    
    ax = plot_df.plot(kind='bar', stacked=True, color=colors, figsize=(12, 7), width=0.6)
    
    plt.title(f"Human vs. AI Judge Alignment - {title_label}", fontsize=14, fontweight='bold', pad=20)
    plt.ylabel("Number of Assessments", fontsize=11)
    plt.xlabel("Evaluated Schema Fields", fontsize=11)
    plt.xticks(rotation=25, ha='right')
    plt.legend(title="Alignment Status", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    # Insert count overlays on populated partitions
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            x, y = p.get_xy() 
            ax.text(
                x + p.get_width() / 2, 
                y + height / 2, 
                f'{int(height)}', 
                ha='center', 
                va='center', 
                color='white', 
                fontweight='bold',
                fontsize=10
            )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_convergence(jd_alignment, title_label="Job Description: alignment between human and judge")
plot_convergence(cv_alignment, title_label="CV: alignment between human and judge")
plot_convergence(matcher_alignment, title_label="Matcher: alignment between human and judge")



In [12]:
model = OpenAIChatModel(
    model_id,
    provider=OpenAIProvider(
        base_url= os.getenv("OPENAI_API_BASE"),
        api_key=os.getenv("OPENAI_API_KEY"), 
    ),
    profile=ModelProfile(
        default_structured_output_mode='json',
        supports_json_schema_output=True,  # tells PydanticAI to send the schema
    ),
)
extra_body_dict = json.loads(os.getenv("LITELLM_EXTRA_BODY"))
settings =  ModelSettings(
    extra_body=extra_body_dict
)

In [32]:

# Path to your MCP server script
script_path = os.path.abspath("mcp_server.py")

# Create a toolset that connects to your running server
toolset = MCPToolset(script_path)


extraction_agent = Agent(
    model,
    model_settings=settings,
    #system_prompt=new_template,
    #deps_type=ArticleReviewInput,
    #output_type=ArticleReviewOutput,
    toolsets=[toolset],
    retries=3 
)


In [28]:
conversation_history = []

In [29]:
result1 = await extraction_agent.run(
    "Please look up the job description for systems_engineer.",
    #allow_tools=True
)

conversation_history = result1.all_messages()
print(result1.output)


The **Systems Engineer** in this automotive context serves as the logical architect of the vehicle, bridging the gap between mechanical design, electrical hardware, and software development. Their primary mission is to manage the complexity of vehicle features (e.g., adaptive cruise control, steering) by decomposing them into functional architectures across various Electronic Control Units (ECUs).

### Key Responsibilities & Workflow
*   **Requirement Management:** Authoring system requirements specifications and designing Interface Control Documents (ICDs) to define signal flow between ECUs.
*   **Architecture & Design:** Modeling functional behavior of subsystems and executing the standardized systems development lifecycle (**V-Model**).
*   **Safety & Compliance:** Collaborating with safety engineers on Hazard Analysis and Risk Assessments (HARA) and drafting functional safety concepts.
*   **Verification:** Working with test engineers to ensure physical system integration matches s

In [30]:
conversation_history

[ModelRequest(parts=[UserPromptPart(content='Please look up the job description for systems_engineer.', timestamp=datetime.datetime(2026, 5, 30, 16, 39, 4, 109396, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 30, 16, 39, 4, 109396, tzinfo=datetime.timezone.utc), run_id='019e79c0-e0a5-76c6-afc0-03817d7a54de', conversation_id='019e79c0-e0a5-76c6-afc0-03805a301d42'),
 ModelResponse(parts=[ToolCallPart(tool_name='get_job_description', args='{"job_key":"systems_engineer"}', tool_call_id='ikawnVlcaXyR0U2biYxu5oigfeXemLuo')], usage=RequestUsage(input_tokens=279, output_tokens=21), model_name='ggml-org/gemma-4-31B-it-GGUF:Q4_K_M', timestamp=datetime.datetime(2026, 5, 30, 16, 39, 5, 174872, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url='https://apphubai.wolke.uni-greifswald.de/v1/', provider_details={'finish_reason': 'tool_calls', 'timestamp': datetime.datetime(2026, 5, 30, 16, 39, 3, tzinfo=TzInfo(0))}, provider_response_id='chatcmpl-0gZrwQkxpnane

In [ ]:
# --- TURN 2 ---
# Ask a follow-up question. The agent now has context of what it did in Turn 1
# because we pass 'conversation_history' back into 'message_history'.
result2 = await extraction_agent.run(
    "What are the non-negotiable toolchains for that role based on what you just looked up?",
    message_history=conversation_history
)

In [ ]:
# =====================================================================
# 4. Define MCP Toolsets (Subprocess / STDIO Transport)
# =====================================================================
# Runs 'server.py' in a managed background process communicating over stdio.
# Ensure the path pointing to server.py is accurate relative to this script.


# =====================================================================
# 5. Initialize Pydantic AI Agent
# =====================================================================
extraction_agent = Agent(
    model,
    model_settings=settings,
    deps_type=ArticleReviewInput,
    output_type=ArticleReviewOutput,
    # Register the local PDF scraping function as a tool
    tools=[scrape_pdf_content],
    # Register the external MCP server's tools
    toolsets=[mcp_toolset],
    retries=3 
)

In [ ]:
from pydantic_ai.mcp import MCPServerStdio

# Path to your MCP server script
script_path = os.path.abspath("mcp_server.py")

# Create a robust, long-lived client connection using 10-minute limits
mcp_server = MCPServerStdio(
    'python', 
    [script_path],
    timeout=600.0,       # Max time in seconds to wait for initial handshake
    read_timeout=600.0   # Max time in seconds to wait for tool responses before dropping
)

extraction_agent = Agent(
    model,
    model_settings=settings,
    toolsets=[mcp_server],  # Use the configured mcp_server instead of the default toolset wrapper
    retries=3 
)

In [3]:
from orchestrator import run_orchestrator_chat


# The user's initial request
user_prompt = """ '
    I have a candidate CV located at 'cv/data/ENGINEERING/12011623.pdf'. Please parse this profile, search for 2 matching jobs remotely or in Germany, 
    evaluate them, and give me a ranked comparison table of the best fits.
"""

In [4]:
result = await run_orchestrator_chat(user_prompt = user_prompt)
result

UnsupportedOperation: fileno

In [ ]:
# Create the official OpenAI client for the sub-agents
mcp_provider = OpenAIProvider(
    base_url=os.getenv("OPENAI_API_BASE"),
    api_key=os.getenv("OPENAI_API_KEY"), 
    http_client=subagent_debug_client  # <--- CORRECT PLACEMENT
)

model = OpenAIChatModel(
    model_id,
    provider=mcp_provider, # <--- Pass the configured provider to the model
    profile=ModelProfile(
        default_structured_output_mode='tool',
        supports_json_schema_output=False, 
    ),
)